In [ ]:
import os
import re
import json
import time
import math
import hashlib
import subprocess
import datetime
import shutil
from typing import List, Dict, Any

import pandas as pd


def find_executable(name: str) -> str:
    path = shutil.which(name)
    return path if path else name


YTDLP_PATH = find_executable("yt-dlp")
FFMPEG_PATH = find_executable("ffmpeg")
FFPROBE_PATH = find_executable("ffprobe")

print(f"[PATH] yt-dlp={YTDLP_PATH}, ffmpeg={FFMPEG_PATH}, ffprobe={FFPROBE_PATH}")

RAW_DIR = "data/raw"
LOG_DIR = "logs"
DATA_DIR = "data"
RESULT_CSV = os.path.join(DATA_DIR, "selector_results.csv")

for d in (RAW_DIR, LOG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

SELECTOR_VERSION = "v0.6"
DUR_MIN, DUR_MAX = 120, 420
SEARCH_TOPN = 5
MAX_RETRY = 2
BACKOFF_SEC = 4

NEG_KEYWORDS = [
    "live", "concert", "cover", "playlist", "full album",
    "lyrics", "sped up", "slowed", "8d", "nightcore",
    "remix", "mashup", "shorts", "teaser", "reaction",
    "audio spectrum", "bass boosted"
]

W_VIEWS = 1.0
W_RECENCY = 0.6
W_TITLE = 0.3


def sha1_hex(text: str) -> str:
    return hashlib.sha1(text.encode()).hexdigest()[:12]


def normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")


def build_uid(title: str, artist: str, vid: str) -> str:
    return sha1_hex(f"yt::{normalize_text(title)}::{normalize_text(artist)}::{vid}")


def run_cmd(cmd: str, capture=False) -> tuple:
    try:
        proc = subprocess.run(
            cmd,
            shell=True,
            capture_output=capture,
            text=True,
            timeout=300
        )
        out = proc.stdout if capture else ""
        err = proc.stderr if capture else ""
        return proc.returncode, out, err
    except subprocess.TimeoutExpired:
        return -1, "", "Timeout"
    except Exception as e:
        return -1, "", str(e)


def download_wav(url: str, out_no_ext: str) -> bool:
    out_tpl = f"{out_no_ext}.%(ext)s"
    cmd = (
        f"\"{YTDLP_PATH}\" --cookies cookies.txt "
        f"--ffmpeg-location \"{os.path.dirname(FFMPEG_PATH)}\" "
        f"-f bestaudio --extract-audio --audio-format wav "
        f"-o \"{out_tpl}\" {url}"
    )
    for i in range(1, MAX_RETRY + 1):
        print(f"[DL] <{i}/{MAX_RETRY}> {url}")
        code, _, err = run_cmd(cmd, capture=True)
        if code == 0 and os.path.exists(out_no_ext + ".wav"):
            return True
        print(f"  ↳ fail: {err.strip()}")
        time.sleep(BACKOFF_SEC * i)
    return False


def probe_duration(path_wav: str) -> int:
    cmd = (
        f"\"{FFPROBE_PATH}\" -v error "
        f"-show_entries format=duration "
        f"-of default=nk=1:nw=1 \"{path_wav}\""
    )
    code, out, _ = run_cmd(cmd, capture=True)
    if code == 0 and out.strip():
        return int(round(float(out.strip())))
    return -1


def yt_search(query: str, topn=SEARCH_TOPN) -> List[Dict[str, Any]]:
    cmd = f"\"{YTDLP_PATH}\" --cookies cookies.txt \"ytsearch{topn}:{query}\" --dump-json --skip-download"
    code, out, err = run_cmd(cmd, capture=True)
    if code != 0:
        print(f"[ERR] search: {err}")
        return []
    result = []
    for line in out.strip().splitlines():
        try:
            result.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return result


def is_negative(title: str) -> bool:
    t = title.lower()
    return any(k in t for k in NEG_KEYWORDS)


def score(meta: Dict[str, Any]) -> float:
    views = meta.get("view_count") or 0
    upload_date = meta.get("upload_date")
    recency_score = 0
    if upload_date:
        try:
            dt = datetime.datetime.strptime(upload_date, "%Y%m%d")
            days = (datetime.datetime.now() - dt).days
            recency_score = max(0, 1 - math.log1p(days) / 10)
        except:
            pass
    clean = 0 if is_negative(meta.get("title", "")) else 1
    return W_VIEWS * math.log1p(views) + W_RECENCY * recency_score + W_TITLE * clean


def select_and_download(track: str, artist: str = "") -> Dict[str, Any]:
    query = f"{track} {artist}".strip()
    print(f"[PROC] {query}")
    meta_list = yt_search(query)
    if not meta_list:
        print("  → no candidates")
        return {}

    filtered = [
        m for m in meta_list
        if DUR_MIN <= (m.get("duration") or 0) <= DUR_MAX
        and not is_negative(m.get("title", ""))
    ]
    if not filtered:
        print("  → none valid")
        return {}

    best = sorted(filtered, key=score, reverse=True)[0]
    vid = best["id"]
    title = best["title"]
    uid = build_uid(title, artist, vid)
    out_no_ext = os.path.join(RAW_DIR, uid)

    ok = download_wav(f"https://www.youtube.com/watch?v={vid}", out_no_ext)
    duration_real = probe_duration(out_no_ext + ".wav") if ok else -1

    return {
        "uid": uid,
        "title": title,
        "artist": artist,
        "video_id": vid,
        "url": f"https://www.youtube.com/watch?v={vid}",
        "duration_meta": best.get("duration"),
        "duration_real": duration_real,
        "views": best.get("view_count"),
        "upload_date": best.get("upload_date"),
        "selector_version": SELECTOR_VERSION,
        "download_ok": ok,
    }


if __name__ == "__main__":
    seed = [
        ("Them Changes", "Thundercat"),
        ("Le Freak", "CHIC"),
        ("Another One Bites the Dust", "Queen"),
        ("Feel Good Inc", "Gorillaz"),
        ("Give It Away", "Red Hot Chili Peppers"),
    ]
    recs = []
    for t, a in seed:
        r = select_and_download(t, a)
        if r:
            recs.append(r)

    if recs:
        df = pd.DataFrame(recs)
        df.to_csv(RESULT_CSV, index=False)
        print(f"[DONE] {len(recs)} tracks → {RESULT_CSV}")
    else:
        print("[DONE] no downloads")


[PATHS] yt-dlp: C:\Users\ghwns\anaconda3\Scripts\yt-dlp.EXE
[PATHS] ffmpeg: C:\Program Files\ffmpeg-7.1.1-essentials_build\bin\ffmpeg.EXE
[PATHS] ffprobe: C:\Program Files\ffmpeg-7.1.1-essentials_build\bin\ffprobe.EXE

[PROCESSING] Them Changes Thundercat
[CMD] "C:\Users\ghwns\anaconda3\Scripts\yt-dlp.EXE" --cookies cookies.txt "ytsearch5:Them Changes Thundercat" --dump-json --skip-download
[DOWNLOAD] try 1/2
[CMD] "C:\Users\ghwns\anaconda3\Scripts\yt-dlp.EXE" --cookies cookies.txt --ffmpeg-location "C:\Program Files\ffmpeg-7.1.1-essentials_build\bin" -f bestaudio --extract-audio --audio-format wav -o "data/raw\bdc7f4f5279c.%(ext)s" https://www.youtube.com/watch?v=BuzJ5NArvgw
[SUCCESS] data/raw\bdc7f4f5279c.wav
[CMD] "C:\Program Files\ffmpeg-7.1.1-essentials_build\bin\ffprobe.EXE" -v error -show_entries format=duration -of default=nk=1:nw=1 "data/raw\bdc7f4f5279c.wav"

[PROCESSING] Le Freak CHIC
[CMD] "C:\Users\ghwns\anaconda3\Scripts\yt-dlp.EXE" --cookies cookies.txt "ytsearch5:Le Fre